# 3.6 算子性能优化

本节总结 Ascend C 算子开发中最通用、最有效的几种优化手段。

---

## 1. Double Buffer / N-Buffer

**原理**：用多份 buffer 实现数据搬运与计算的重叠。

<img src="./images/double_buffer.png" alt="double buffer" width="700px">

实现：
```cpp
constexpr uint32_t BUFFER_NUM = 2;
pipe.InitBuffer(inQueueX, BUFFER_NUM, tileLength * sizeof(float));
pipe.InitBuffer(outQueueY, BUFFER_NUM, tileLength * sizeof(float));
// 循环次数 = tileNum × BUFFER_NUM，自动轮转
for (int32_t i = 0; i < tileNum * BUFFER_NUM; i++) {
    CopyIn(i); Compute(i); CopyOut(i);
}
```

当 UB 空间充足时，可以增加到 BUFFER_NUM=4 甚至更高，进一步加深流水线：
```cpp
while (perLoopElements > singleCopyElements && bufferNum < MAX_BUFFER_NUM) {
    bufferNum++;
    perLoopElements = (ubSize - fixedCost) / perElementBytes / bufferNum;
}
```

---

## 2. UB 空间优化

**原理**：UB 容量固定（256KB），合理利用能减少循环次数、降低标量开销。

### 2.1 tileNum 调优

增大 tileNum → tileLength 减小 → 单次 UB 占用减小 → 但循环次数增多。
需要在 UB 容量和循环开销之间找到平衡：

```
UB 占用 = (N_input × BUFFER_NUM + N_output × BUFFER_NUM + N_temp) × tileLength × sizeof(T)
tileLength ≤ UB_SIZE / ((N_input + N_output) × BUFFER_NUM + N_temp) / sizeof(T)
```

### 2.2 多行合并

当每行数据量远小于 UB 容量时，将多行合并到一个 UB 批处理中，减少循环次数：

```cpp
// 每行的 UB 开销 = 固定开销 + 线性开销 × ubFactor
// ubFactor 越大，循环次数越少
int64_t maxUbFactor = (ubSize - fixedSize) / linearCoefPerRow;
// MTE2 一次 DMA 搬运 ubFactor 行
DataCopyExtParams.blockCount = ubFactor;
```

---

## 3. Scalar 优化

**原理**：Scalar 单元向所有流水线（Cube、Vector、MTE）派发指令。当 Scalar 成为瓶颈时，其他流水线全部空转。Scalar 瓶颈的根因通常是**寄存器溢出**——编译器将变量溢写到堆栈，导致额外的 Load/Store 指令。

| 原则 | 错误写法 | 正确写法 |
|------|---------|----------|
| **用局部变量** | 频繁访问 `this->member_` | 在局部变量中缓存 |
| **就近定义** | 函数顶部统一声明 | 使用前才定义，缩短 live range |
| **避免指针多层解引用** | `ptr_->ptr_->value` | 使用值类型或局部缓存 |
| **避免大结构体** | struct > 64 bytes | 拆分或用引用传递 |
| **热路径去分支** | 循环内有 if/else | 将分支移出循环 |

实测效果：Scalar 指令减少 33-50%，端到端性能提升 3-27%。

---

## 4. 数据类型选择

**原理**：不同数据类型的位宽和吞吐不同。选择正确的类型可以降低显存带宽压力或提高精度。

| 类型 | 位宽 | 每次操作元素数 | 适用场景 |
|------|------|------|------|
| `half` (FP16) | 16-bit | 最多 | 推理、大数据量 |
| `bfloat16_t` (BF16) | 16-bit | 最多 | 训练、需保留 FP32 指数范围 |
| `float` (FP32) | 32-bit | F16 的一半 | 训练、高精度需求 |
| `int8_t` / `uint8_t` | 8-bit | F32 的四倍 | 量化推理 |

- FP16 计算时，中间结果可能需上转 FP32 避免精度损失（`Cast<float, half>`）
- BF16 保留与 FP32 相同的指数位宽，梯度计算更稳定
- 输入输出用小位宽、中间计算用大位宽是常见的精度/性能折中方案

---

## 5. 数据对齐

**原理**：对齐访问比非对齐访问快。`LoadAlign` / `StoreAlign` 利用硬件对齐加速，`DataCopy` 处理非对齐场景。

```cpp
// 对齐访问（快）—— 要求地址对齐到向量边界
LoadAlign(reg, alignedAddr);
StoreAlign(alignedAddr, reg, mask);

// 非对齐访问（慢）—— 兜底方案
DataCopy(reg, unalignedAddr, len);
```

对齐规则：
- 搬运前先用 `GetVecLen()` 计算向量长度
- `tileLength` 尽量取向量长度的整数倍
- 最后一块不足对齐时用非对齐接口兜底

---

## 6. 多核并行

**原理**：将数据均分到所有 AI Core 上并行处理，是最直接有效的加速手段。

### 6.1 基础分核

```cpp
uint32_t blockLength = totalLength / GetBlockNum();
uint32_t offset = blockLength * GetBlockIdx();
xGm.SetGlobalBuffer((__gm__ float*)x + offset, blockLength);
```

### 6.2 尾块均衡

当最后一轮剩余 tile 数少于核数时，部分核会空闲。将剩余 tile 进一步拆分子块，均分到更多核上：

```cpp
// 在 M 和 N 维度上贪心拆分，最大化尾块轮的核心利用率
while ((tailMCnt + 1) * tailNCnt * tailCnt <= coreNum) {
    tailMCnt++;  // 先尝试 M 维度拆分
    if (tailMCnt * (tailNCnt + 1) * tailCnt <= coreNum)
        tailNCnt++;  // 再尝试 N 维度拆分
}
```

---

## 7. 带宽利用率

**原理**：单次搬运数据量越大，HBM 带宽利用率越高。避免零散的细碎搬运。

```cpp
// ❌ 多次小块搬运：带宽利用率低
for (int i = 0; i < N; i++) {
    DataCopy(dst + i * smallLen, src + i * smallLen, smallLen);
}

// ✅ 合并没有依赖的多次搬运为一次大搬运
DataCopy(dst, src, N * smallLen);
```

对于不同维度的数据（如行和列均有切分），优先将大维度方向合并到单次搬运中。

---

## 小结

| 优化手段 | 核心思路 | 典型收益 |
|------|------|------|
| Double Buffer | 搬运与计算并行 | ~40% |
| UB 空间优化 | 增大 tileLength 减少循环 | 标量指令 -56% |
| Scalar 优化 | 减少寄存器溢出 | 3-27% |
| 数据类型选择 | FP16/BF16 提高吞吐 | 吞吐 ×2（vs FP32） |
| 数据对齐 | LoadAlign 对齐加速 | 显著 |
| 多核并行 | 尾块均衡提高核利用率 | 最高（×核数） |
| 带宽利用 | 合并零散搬运 | 显著 |

---

上一节：[3.3 调用接口开发](./03.04_api_interface_development.ipynb)